
# LITERATURE RADAR — V1

In [20]:
# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import requests
import pandas as pd

from bs4 import BeautifulSoup
from openai import OpenAI
from urllib.parse import quote, urljoin
from getpass import getpass

In [6]:
# ------------------------------------------------------------
# 2. ENTER YOUR OPENAI API KEY
# ------------------------------------------------------------

# The key is entered interactively and is not written into
# the notebook.

# skip if you are importing your API key from repo clone
# if not, the API key should be entered here
'''
api_key = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=api_key)

print("✓ OpenAI client ready")
'''

'\napi_key = getpass("Enter your OpenAI API key: ")\n\nclient = OpenAI(api_key=api_key)\n\nprint("✓ OpenAI client ready")\n'

In [45]:
# ------------------------------------------------------------
# 2. CREATE LLM CLIENTS
# ------------------------------------------------------------

import os
from dotenv import load_dotenv
from openai import OpenAI

# Load OpenAI API key from .env
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ No OpenAI API key was found.")
elif not api_key.startswith("sk-proj-"):
    print("⚠️ API key found, but it doesn't look like an OpenAI project key.")
else:
    print("✓ OpenAI API key found.")

# Frontier LLM
openai_client = OpenAI(api_key=api_key)

# Local LLM through Ollama's OpenAI-compatible endpoint
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

✓ OpenAI API key found.


In [40]:
# add Ollama endpoint
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [52]:
# ------------------------------------------------------------
# 3. DEFINE YOUR SEARCH KEYWORD
# ------------------------------------------------------------

keyword = "self driving lab thermal catalysis"

print(f"Search keyword: {keyword}")

Search keyword: self driving lab thermal catalysis


In [54]:
# ------------------------------------------------------------
# 4. SEARCH NATURE
# ------------------------------------------------------------

search_url = f"https://www.nature.com/search?q={quote(keyword)}&order=relevance"

print("Searching Nature...")
print(search_url)

search_response = requests.get(search_url)

print("Status code:", search_response.status_code)

search_soup = BeautifulSoup(
    search_response.text,
    "html.parser"
)

Searching Nature...
https://www.nature.com/search?q=self%20driving%20lab%20thermal%20catalysis&order=relevance
Status code: 200


In [55]:
# ------------------------------------------------------------
# 5. EXTRACT ARTICLE LINKS
# ------------------------------------------------------------

article_links = []

for link in search_soup.find_all("a", href=True):

    href = link["href"]
    title = link.get_text(strip=True)

    # Keep Nature article URLs
    if href.startswith("/articles/") and title:

        full_url = urljoin(
            "https://www.nature.com",
            href
        )

        article_links.append({
            "title": title,
            "url": full_url
        })


print(f"Found {len(article_links)} articles")


Found 50 articles


In [56]:
# ------------------------------------------------------------
# 6. DISPLAY FOUND ARTICLES
# ------------------------------------------------------------

for i, article in enumerate(article_links, start=1):

    print(f"{i}. {article['title']}")
    print(f"   {article['url']}")
    print()

1. Autonomous reaction Pareto-front mapping with a self-driving catalysis laboratory
   https://www.nature.com/articles/s44286-024-00033-5

2. Ambient solar thermal catalysis for polyolefin upcycling using copper encapsulated in silicon nanosheets and chloroaluminate ionic liquid
   https://www.nature.com/articles/s41929-025-01349-y

3. Autonomous catalysis research with human–AI–robot collaboration
   https://www.nature.com/articles/s41929-025-01430-6

4. The past, present and future of self-driving laboratories
   https://www.nature.com/articles/s41570-026-00847-2

5. Towards self-driving laboratories in the biopharmaceutical industry
   https://www.nature.com/articles/s44160-026-01131-3

6. The paradox of thermal vs. non-thermal effects in plasmonic photocatalysis
   https://www.nature.com/articles/s41467-024-51916-3

7. Language models and protocol standardization guidelines for accelerating synthesis planning in heterogeneous catalysis
   https://www.nature.com/articles/s41467-023

In [57]:
# ------------------------------------------------------------
# 7. FUNCTION TO FETCH ARTICLE TEXT
# ------------------------------------------------------------

def fetch_article_text(url):

    response = requests.get(url)

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove webpage elements that are not useful
    # for scientific analysis
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()

    text = soup.get_text(
        separator="\n",
        strip=True
    )

    return text

In [14]:
# ------------------------------------------------------------
# 8. TEST ARTICLE SCRAPING
# ------------------------------------------------------------
'''
test_text = fetch_article_text(
    article_links[0]["url"]
)

print(test_text[:3000])
'''

'\ntest_text = fetch_article_text(\n    article_links[0]["url"]\n)\n\nprint(test_text[:3000])\n'

In [58]:
# ------------------------------------------------------------
# 9. DEFINE YOUR SYSTEM PROMPT
# ------------------------------------------------------------

system_prompt = """
You are a scientific literature assistant with expertise in chemistry,
catalysis, materials science, and research data management.

Analyze the provided scientific publication carefully and objectively.

Only use information contained in the supplied publication text.
Do not invent or infer information that is not supported by the text.
If requested information is not available, clearly state that it is not available.
"""


In [59]:
# ------------------------------------------------------------
# 10. DEFINE YOUR USER PROMPT
# ------------------------------------------------------------

user_prompt = """
For this publication, provide:

- publication year
- first author
- journal
- article type
- a 2-3 sentence summary
- why the publication is relevant to my search keyword
"""


In [60]:
# ------------------------------------------------------------
# 11. FUNCTION TO ANALYZE ONE ARTICLE WITH THE LLM
# ------------------------------------------------------------

def analyze_article(article, keyword, llm_client, model):

    article_text = fetch_article_text(
        article["url"]
    )

    full_user_prompt = f"""
Search keyword: {keyword}

Your task:
{user_prompt}

Publication title:
{article["title"]}

Publication URL:
{article["url"]}

Publication webpage text:
{article_text[:20000]}
"""

    response = llm_client.chat.completions.create(

        model=model,

        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": full_user_prompt
            }
        ]
    )

    return response.choices[0].message.content

In [61]:
# ------------------------------------------------------------
# 12. CHOOSE LLM
# ------------------------------------------------------------

# Choose provider:
# "openai"  → frontier model
# "ollama"  → local model

# provider = "ollama"
provider = 'openai'

# Choose model:
# model = "llama3.2"
model = "gpt-5-mini"


if provider == "openai":

    llm_client = openai_client

elif provider == "ollama":

    llm_client = ollama_client

else:

    raise ValueError(
        f"Unknown provider: {provider}"
    )


print(f"✓ Selected provider: {provider}")
print(f"✓ Selected model: {model}")

✓ Selected provider: openai
✓ Selected model: gpt-5-mini


In [62]:
# ------------------------------------------------------------
# 12 TEST THE LLM ON ONE ARTICLE
# ------------------------------------------------------------

test_result = analyze_article(
    article_links[0],
    keyword,
    llm_client,
    model
)

print(test_result)

- Publication year: 2024 (Published 27 February 2024)
- First author: J. A. Bennett
- Journal: Nature Chemical Engineering
- Article type: Article

Summary (2–3 sentences):
This work presents Fast‑Cat, a fully autonomous self‑driving catalysis laboratory built around a modular gas–liquid flow platform with in‑line GC analysis and a machine‑learning “brain” (an ensemble DNN + Bayesian acquisition via qNEHVI). Fast‑Cat autonomously explored high‑temperature, high‑pressure hydroformylation of 1‑octene with rhodium/phosphorus ligands to rapidly identify Pareto fronts (tradeoffs between total aldehyde yield and linear/branched regioselectivity), generating high‑quality experimental data and a digital twin over 45 reactions in 5 days without human intervention.

Why this is relevant to your search ("self driving lab thermal catalysis"):
The paper describes a self‑driving laboratory applied to homogeneous catalytic reactions that operate under elevated temperature and pressure (high‑temperatu

In [64]:
# ------------------------------------------------------------
# 13. ANALYZE ALL ARTICLES
# ------------------------------------------------------------

results = []

for i, article in enumerate(
    article_links,
    start=1
):

    print(
        f"Analyzing {i}/{len(article_links)}: "
        f"{article['title']}"
    )

    analysis = analyze_article(
        article,
        keyword,
        llm_client,
        model
    )

    results.append({

        "title": article["title"],

        "url": article["url"],

        "analysis": analysis

    })


print("\n✓ Analysis complete!")

Analyzing 1/50: Autonomous reaction Pareto-front mapping with a self-driving catalysis laboratory
Analyzing 2/50: Ambient solar thermal catalysis for polyolefin upcycling using copper encapsulated in silicon nanosheets and chloroaluminate ionic liquid
Analyzing 3/50: Autonomous catalysis research with human–AI–robot collaboration
Analyzing 4/50: The past, present and future of self-driving laboratories
Analyzing 5/50: Towards self-driving laboratories in the biopharmaceutical industry
Analyzing 6/50: The paradox of thermal vs. non-thermal effects in plasmonic photocatalysis
Analyzing 7/50: Language models and protocol standardization guidelines for accelerating synthesis planning in heterogeneous catalysis
Analyzing 8/50: Embracing data science in catalysis research
Analyzing 9/50: Reaction environment impacts charge transfer but not chemical reaction steps in hydrogen evolution catalysis
Analyzing 10/50: Autonomous discovery of optically active chiral inorganic perovskite nanocrystals

In [65]:
# ------------------------------------------------------------
# 14. CREATE PANDAS TABLE
# ------------------------------------------------------------

df = pd.DataFrame(results)


# Rename columns
df = df.rename(columns={

    "title": "Publication",

    "url": "URL",

    "analysis": "AI Analysis"

})


# Put columns in desired order
df = df[
    [
        "Publication",
        "AI Analysis",
        "URL"
    ]
]




In [66]:
# ------------------------------------------------------------
# 15. DISPLAY COMPLETE AI ANALYSIS
# ------------------------------------------------------------

# Prevent pandas from truncating long text
pd.set_option(
    "display.max_colwidth",
    None
)

display(df)

# save results
df.to_excel(f'{keyword}.xlsx')

,Publication,AI Analysis,URL
0,Autonomous reaction Pareto-front mapping with a self-driving catalysis laboratory,"- Publication year: 2024 (Published 27 February 2024)\n- First author: J. A. Bennett\n- Journal: Nature Chemical Engineering\n- Article type: Article\n\n2–3 sentence summary:\nThe paper presents Fast‑Cat, a self‑driving catalysis laboratory that couples a modular gas–liquid flow reactor, in‑line GC characterization and an ensemble DNN + Bayesian optimization workflow (qNEHVI) to autonomously map multi‑objective Pareto fronts for high‑temperature, high‑pressure homogeneous reactions. The authors demonstrate autonomous ligand benchmarking and rapid Pareto‑front identification for Rh‑catalyzed hydroformylation of 1‑octene (six ligand campaigns; 45 reactions run continuously over 5 days), and show the platform can generate high‑quality data for digital twins and scale insights from miniaturized flow to batch reactors.\n\nWhy this publication is relevant to your search ""self driving lab thermal catalysis"":\nThis work describes a self‑driving laboratory specifically built for catalysis: an autonomous, closed‑loop flow platform for high‑temperature/high‑pressure gas–liquid (thermal) homogeneous catalytic reactions, using machine learning to select experiments and optimize multi‑objective performance (yield vs regioselectivity). Its focus on autonomous exploration, Pareto‑front mapping, and generation of scalable reaction data for thermal catalytic processes makes it directly relevant to studies of self‑driving labs applied to thermal catalysis.",https://www.nature.com/articles/s44286-024-00033-5
1,Ambient solar thermal catalysis for polyolefin upcycling using copper encapsulated in silicon nanosheets and chloroaluminate ionic liquid,"- Publication year: 2025 (Published 05 June 2025)\n- First author: Chuanwang Xing\n- Journal: Nature Catalysis\n- Article type: Article\n\n2–3 sentence summary:\nThis paper reports a solar-thermal catalytic system that upcycles polyolefins (LDPE) at a mild photothermally-generated temperature (55 °C under 4 sun illumination) using copper nanoparticles encapsulated in stacked two-dimensional silicon (Cu/2D Si) in a chloroaluminate ionic-liquid/chloroform solvent. The system converts LDPE to separable fractions of alkanes (C3–C7) and cyclic hydrocarbons (C8–C26) with a total yield of 91% within 6 hours; mechanistic studies indicate two β‑scissions of C–C bonds followed by rapid intramolecular cyclization, and techno‑economic and LCA analyses show greenhouse‑gas reductions and possible economic viability.\n\nWhy this publication is relevant to your search keyword (""self driving lab thermal catalysis""):\n- It directly concerns thermal catalysis (specifically photothermal/solar-thermal catalysis) for chemical transformation of plastics, describing catalyst design, reaction conditions, product distributions and mechanistic insight—materially relevant to the ""thermal catalysis"" part of your query. \n- The article does not mention or describe self-driving laboratories or autonomous experimental platforms; no information about automated or closed‑loop/self‑driving laboratory implementation appears in the supplied text.",https://www.nature.com/articles/s41929-025-01349-y
2,Autonomous catalysis research with human–AI–robot collaboration,"- Publication year: 2025\n- First author: Negin Orouji\n- Journal: Nature Catalysis\n- Article type: Perspective\n\n2–3 sentence summary:\nThis Perspective describes the integration of AI, robotics and high-throughput experimentation into self-driving laboratories (SDLs) for accelerating catalyst discovery and optimization. It outlines core SDL components (automation hardware, data infrastructure, computational modelling and AI-guided decision-making), highlights challenges such as limited/poor-quality data and the need for human oversight to validate machine-generated hypotheses, and discusses domain-specific obstacles (e.g., automating experiments under high pre